## Install Dependencies

Installing the libraries needed across all experiments: `scikit-learn` for TF-IDF retrieval,
`rouge-score` for local evaluation, and `transformers`/`sentencepiece` for the mT5 fine-tuning
experiments later in this notebook.

In [ ]:
!pip install scikit-learn pandas numpy rouge-score transformers sentencepiece -q
print("Done")

  Preparing metadata (setup.py) ... done
Done


## Mount Google Drive

Mounting Drive to access the competition zip file and to save/load model checkpoints and
submissions, so nothing is lost if the Colab runtime disconnects.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports & Reproducibility Setup

Core imports for data handling and TF-IDF retrieval. Seed is fixed at 42 across `random` and
`numpy` so results are reproducible — a requirement for the competition's code review.

In [6]:
import re, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports done")

ModuleNotFoundError: No module named 'rouge_score'

## Load Competition Data

Extracting the Zindi competition zip and loading Train, Val, and Test sets. Train (29,815 rows)
and Val (6,686 rows) include both questions and reference answers; Test (2,618 rows) has
questions only — this is what we generate predictions for.

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/multilingual-health-question-answering-in-low-resource-african-languages-challenge20260521-4309-nrr2a2.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/data/')

train = pd.read_csv('/content/data/Train.csv')
val   = pd.read_csv('/content/data/Val.csv')
test  = pd.read_csv('/content/data/Test.csv')

print("Train:", train.shape, "| Val:", val.shape, "| Test:", test.shape)

Train: (29815, 4) | Val: (6686, 4) | Test: (2618, 3)


## Helper Functions

Reusable utilities for every experiment below:
- `LANGUAGE_MAP` resolves subset codes (e.g. `Amh_Eth`) to readable language names
- `compute_rouge` / `compute_rouge_by_language` use a whitespace tokenizer instead of the
  default, since standard tokenizers aren't reliable on non-Latin scripts like Amharic
- `make_submission` builds the 4-column submission file, strips any leftover `<extra_id_N>`
  sentinel tokens, and validates row count/column alignment before saving

In [ ]:
# Language mapping
LANGUAGE_MAP = {
    "Aka_Gha": "Akan", "Amh_Eth": "Amharic",
    "Eng_Eth": "English", "Eng_Gha": "English",
    "Eng_Ken": "English", "Eng_Uga": "English",
    "Lug_Uga": "Luganda", "Swa_Ken": "Swahili",
}

# ROUGE scorer with whitespace tokenizer (safe for African scripts)
class WhitespaceTokenizer:
    def tokenize(self, text):
        return str(text).strip().split() if text else []

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        tokenizer=WhitespaceTokenizer(),
        use_stemmer=False
    )
    r1, rl = [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(str(ref), str(pred))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {'rouge1_f1': float(np.mean(r1)), 'rougeL_f1': float(np.mean(rl))}

def compute_rouge_by_language(predictions, references, languages):
    results = {}
    lang_arr = np.array(languages)
    for lang in np.unique(lang_arr):
        mask = lang_arr == lang
        results[lang] = compute_rouge(
            [p for p, m in zip(predictions, mask) if m],
            [r for r, m in zip(references, mask) if m]
        )
    return pd.DataFrame(results).T

# Submission builder with validation checks
def make_submission(ids, predictions, output_path):
    clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]
    sub = pd.DataFrame({
        'ID': ids,
        'TargetRLF1': clean_preds,
        'TargetR1F1': clean_preds,
        'TargetLLM':  clean_preds,
    })[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
    assert len(sub) == len(test), f"Row mismatch: {len(sub)} vs {len(test)}"
    assert (sub['TargetRLF1'] == sub['TargetR1F1']).all()
    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Saved: {output_path} | Shape: {sub.shape}")
    return sub

print("Helper functions ready")

Helper functions ready


## Experiment 1 — Dummy Baseline

Copies the input question as the answer for every test row. Purpose: validate the submission
pipeline (correct columns, row count, format) before investing time in any real modeling.

**Result:** Submitted successfully. Score not recorded — not meaningful, purpose was format
validation only.

In [ ]:
make_submission(test['ID'].values, test['input'].tolist(), '/content/drive/MyDrive/submission_e1_dummy.csv')

## Experiment 2 — mT5-Small Zero-Shot

Loads raw pretrained `google/mt5-small` with no fine-tuning and generates directly on test
questions. Purpose: establish a lower bound — how does an unmodified pretrained multilingual
model perform with zero adaptation?

**Result:** Outputs were `<extra_id_0>` sentinel tokens followed by input fragments
(e.g. `<extra_id_0> GBV ano ma.`). Score not recorded. mT5 is pretrained on span-corruption —
predicting masked spans — not question answering, so zero-shot it just performs the
pretraining task instead.

In [ ]:
from transformers import AutoTokenizer, MT5ForConditionalGeneration
import torch
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_e2 = AutoTokenizer.from_pretrained("google/mt5-small")
model_e2 = MT5ForConditionalGeneration.from_pretrained("google/mt5-small").to(device)
model_e2.eval()

predictions_e2 = []
for text in tqdm(test['input'].tolist()):
    inputs = tokenizer_e2(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model_e2.generate(**inputs, max_length=256)
    predictions_e2.append(tokenizer_e2.decode(outputs[0], skip_special_tokens=True))

make_submission(test['ID'].values, predictions_e2, '/content/drive/MyDrive/submission_e2_zeroshot.csv')

## Experiment 3 — mT5-Small Fine-Tuned (Greedy Decoding)

Fine-tunes mT5-small on the full training set for 1 epoch, generates with greedy decoding.

**Key challenges resolved:**
- fp16 caused training loss to collapse to 0.0/NaN — fixed by disabling fp16
- OOM at batch_size=8 — fixed with batch_size=4, gradient_accumulation_steps=2
- Colab disconnects lost trained weights multiple times — fixed by saving to Drive immediately after training
- CSV submission errors from unescaped quotes — fixed with `quoting=csv.QUOTE_ALL`

**Training result:** Training loss 6.4, Validation loss 2.279

**Zindi score: 0.168657**

**Output quality:** Real text in correct languages but severe repetition loops
(e.g. "No. No. No.") from greedy decoding getting stuck in high-probability cycles.

> Note: code not re-run in this session due to repeated Colab disconnects during
> debugging — model and submission were saved from the original run, and this score
> motivated the pivot to retrieval-based approaches in Experiments 4 onward.

## Experiment 4 — Global TF-IDF Retrieval

Instead of generating answers, retrieve them: build one TF-IDF index (character 3-5 grams)
across all 29,815 training questions, then return the answer of the nearest match for each
test question. Character n-grams work across all scripts (Latin, Ge'ez) without language-specific
tokenization.

**Result:** ROUGE-1 0.4279 | Zindi 0.497641 — outperforms fine-tuned mT5-small by ~3x.

In [ ]:
# Build global TF-IDF index on all training data
print("Building global TF-IDF index...")

tfidf_global = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=200_000,
    lowercase=False
)
train_vecs = tfidf_global.fit_transform(train['input'].fillna('').tolist())

nn_global = NearestNeighbors(n_neighbors=1, metric='cosine')
nn_global.fit(train_vecs)
print("Global TF-IDF index ready")

# Predict on validation set
def retrieve_global(df):
    vecs = tfidf_global.transform(df['input'].fillna('').tolist())
    _, indices = nn_global.kneighbors(vecs)
    return [train['output'].iloc[idx[0]] for idx in indices]

val_pred_e4 = retrieve_global(val)

metrics_e4 = compute_rouge(val_pred_e3, val['output'].tolist())
print(f"\n Experiment 3 — Global TF-IDF Retrieval")
print(f"   ROUGE-1 F1: {metrics_e3['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e3['rougeL_f1']:.4f}")

print("\nBy language:")
display(compute_rouge_by_language(val_pred_e3, val['output'].tolist(), val['subset'].tolist()).round(4))

# Save submission
test_pred_e3 = retrieve_global(test)
make_submission(test['ID'].values, test_pred_e3, '/content/drive/MyDrive/submission_e3_tfidf_global.csv')

Building global TF-IDF index...
Global TF-IDF index ready

 Experiment 3 — Global TF-IDF Retrieval
   ROUGE-1 F1: 0.4279
   ROUGE-L F1: 0.3743

By language:


,rouge1_f1,rougeL_f1
Aka_Gha,0.2869,0.1693
Amh_Eth,0.1529,0.1433
Eng_Eth,0.5062,0.4879
Eng_Gha,0.2607,0.1740
Eng_Ken,0.6031,0.5694
Eng_Uga,0.5447,0.5059
Lug_Uga,0.5049,0.4814
Swa_Ken,0.6093,0.5735


Saved: /content/drive/MyDrive/submission_e3_tfidf_global.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ..."
1,ID_TS_Aka_Gha_1C80317F,"Yiw, aman pii wɔ mmara anaa nhyehyɛe ahorow a ...","Yiw, aman pii wɔ mmara anaa nhyehyɛe ahorow a ...","Yiw, aman pii wɔ mmara anaa nhyehyɛe ahorow a ..."
2,ID_TS_Aka_Gha_06671AD1,"Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh..."
3,ID_TS_Aka_Gha_BDD640FB,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...
4,ID_TS_Aka_Gha_46685257,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...
2614,ID_TS_Swa_Ken_FD034ADA,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...
2616,ID_TS_Swa_Ken_8DDCE719,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...


## Experiment 5 — Language-Specific TF-IDF

Builds a separate TF-IDF index per language subset (8 indexes total) instead of one global
index, so retrieval only matches within the same language. Hypothesis: language-matched
retrieval gives more linguistically appropriate answers than cross-language matching.

**Result:** ROUGE-1 0.4209 | Zindi 0.447403 — slightly worse than global. Smaller per-language
pools (e.g. only 1,845 examples for Amharic) hurt match quality more than language-matching helps.

In [ ]:
print("Building per-language TF-IDF indexes...")

subset_tfidf, subset_nn, subset_train = {}, {}, {}

for subset in train['subset'].unique():
    data = train[train['subset'] == subset].reset_index(drop=True)
    subset_train[subset] = data

    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=100_000, lowercase=False)
    vecs = vec.fit_transform(data['input'].fillna('').tolist())

    nn = NearestNeighbors(n_neighbors=1, metric='cosine')
    nn.fit(vecs)

    subset_tfidf[subset] = vec
    subset_nn[subset] = nn
    print(f"  {subset}: {len(data)} examples")

print("Per-language indexes ready")

def retrieve_language_specific(df):
    results = [''] * len(df)
    for subset in df['subset'].unique():
        mask = df['subset'] == subset
        subset_rows = df[mask]
        vecs = subset_tfidf[subset].transform(subset_rows['input'].fillna('').tolist())
        _, indices = subset_nn[subset].kneighbors(vecs)
        retrieved = [subset_train[subset].iloc[idx[0]]['output'] for idx in indices]
        for i, (orig_idx, answer) in enumerate(zip(subset_rows.index, retrieved)):
            results[df.index.get_loc(orig_idx)] = answer
    return results

val_pred_e4 = retrieve_language_specific(val)
metrics_e4 = compute_rouge(val_pred_e4, val['output'].tolist())
print(f"\n Experiment 4 — Language-Specific TF-IDF")
print(f"   ROUGE-1 F1: {metrics_e4['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e4['rougeL_f1']:.4f}")

print("\nBy language:")
display(compute_rouge_by_language(val_pred_e4, val['output'].tolist(), val['subset'].tolist()).round(4))

test_pred_e4 = retrieve_language_specific(test)
make_submission(test['ID'].values, test_pred_e4, '/content/drive/MyDrive/submission_e4_tfidf_lang.csv')

Building per-language TF-IDF indexes...
  Aka_Gha: 4455 examples
  Amh_Eth: 1845 examples
  Eng_Eth: 3915 examples
  Eng_Gha: 4443 examples
  Eng_Ken: 2080 examples
  Eng_Uga: 7624 examples
  Lug_Uga: 3383 examples
  Swa_Ken: 2070 examples
Per-language indexes ready

 Experiment 4 — Language-Specific TF-IDF
   ROUGE-1 F1: 0.4208
   ROUGE-L F1: 0.3656

By language:


,rouge1_f1,rougeL_f1
Aka_Gha,0.2832,0.1674
Amh_Eth,0.1455,0.1353
Eng_Eth,0.5170,0.4994
Eng_Gha,0.2582,0.1707
Eng_Ken,0.5989,0.5606
Eng_Uga,0.5165,0.4710
Lug_Uga,0.5155,0.4935
Swa_Ken,0.6031,0.5672


Saved: /content/drive/MyDrive/submission_e4_tfidf_lang.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ..."
1,ID_TS_Aka_Gha_1C80317F,"Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa ..."
2,ID_TS_Aka_Gha_06671AD1,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basab...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basab...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basab...
3,ID_TS_Aka_Gha_BDD640FB,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...
4,ID_TS_Aka_Gha_46685257,Kae sɛ obiara suahunu yɛ soronko. Dwen sɛ obia...,Kae sɛ obiara suahunu yɛ soronko. Dwen sɛ obia...,Kae sɛ obiara suahunu yɛ soronko. Dwen sɛ obia...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...
2614,ID_TS_Swa_Ken_FD034ADA,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...
2616,ID_TS_Swa_Ken_8DDCE719,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...


## Experiment 6 — Top-3 Retrieval, Longest Answer

Retrieves the top-3 nearest neighbors (instead of just 1) and picks the longest answer among
them. Hypothesis: longer answers are more complete and may score better, particularly on
LLM-as-a-Judge which rewards completeness.

**Result:** ROUGE-1 0.3713 | Zindi 0.491549 — lower local ROUGE (longest ≠ most relevant),
but a smaller drop on Zindi than expected, since the LLM judge partially rewards the added length.

In [ ]:
# Rebuild nn with n_neighbors=3
nn_global_3 = NearestNeighbors(n_neighbors=3, metric='cosine')
nn_global_3.fit(train_vecs)

def retrieve_top3_longest(df):
    vecs = tfidf_global.transform(df['input'].fillna('').tolist())
    _, indices = nn_global_3.kneighbors(vecs)
    results = []
    for idx_row in indices:
        candidates = [train['output'].iloc[idx] for idx in idx_row]
        longest = max(candidates, key=lambda x: len(str(x).split()))
        results.append(longest)
    return results

val_pred_e5 = retrieve_top3_longest(val)
metrics_e5 = compute_rouge(val_pred_e5, val['output'].tolist())
print(f"\n Experiment 5 — Top-3 Retrieval, Longest Answer")
print(f"   ROUGE-1 F1: {metrics_e5['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e5['rougeL_f1']:.4f}")

test_pred_e5 = retrieve_top3_longest(test)
make_submission(test['ID'].values, test_pred_e5, '/content/drive/MyDrive/submission_e5_top3_longest.csv')


 Experiment 5 — Top-3 Retrieval, Longest Answer
   ROUGE-1 F1: 0.3710
   ROUGE-L F1: 0.3082
Saved: /content/drive/MyDrive/submission_e5_top3_longest.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,Nneɛma bi a wɔde bɛyɛ intanɛt so nkyerɛkyerɛ a...,Nneɛma bi a wɔde bɛyɛ intanɛt so nkyerɛkyerɛ a...,Nneɛma bi a wɔde bɛyɛ intanɛt so nkyerɛkyerɛ a...
1,ID_TS_Aka_Gha_1C80317F,"Yiw, mmara ne nhyehyɛe ahorow bɔ hokwan ahorow...","Yiw, mmara ne nhyehyɛe ahorow bɔ hokwan ahorow...","Yiw, mmara ne nhyehyɛe ahorow bɔ hokwan ahorow..."
2,ID_TS_Aka_Gha_06671AD1,"Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh..."
3,ID_TS_Aka_Gha_BDD640FB,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...
4,ID_TS_Aka_Gha_46685257,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,Ukimwi (Virusi vya Upungufu wa Kinga Mwilini) ...,Ukimwi (Virusi vya Upungufu wa Kinga Mwilini) ...,Ukimwi (Virusi vya Upungufu wa Kinga Mwilini) ...
2614,ID_TS_Swa_Ken_FD034ADA,Hakuna ushahidi wa kisayansi unaothibitisha ma...,Hakuna ushahidi wa kisayansi unaothibitisha ma...,Hakuna ushahidi wa kisayansi unaothibitisha ma...
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...
2616,ID_TS_Swa_Ken_8DDCE719,Kuishi na mtu ambaye ana Ukimwi/UKIMWI kunahit...,Kuishi na mtu ambaye ana Ukimwi/UKIMWI kunahit...,Kuishi na mtu ambaye ana Ukimwi/UKIMWI kunahit...


## Experiment 7 — Top-3 Retrieval, Most Similar

Same top-3 candidate pool as Experiment 5, but instead of picking the longest answer, picks
the one with the lowest cosine distance (most similar).

**Result:** ROUGE-1 0.4279 | Zindi 0.497641 — identical to Experiment 3. Mathematically, the
most similar of the top-3 is always the same as the top-1 nearest neighbor, so this offers no
benefit over simple top-1 retrieval.

In [ ]:
# Instead of longest, pick the most similar answer
nn_global_3_scores = NearestNeighbors(n_neighbors=3, metric='cosine')
nn_global_3_scores.fit(train_vecs)

def retrieve_top3_most_similar(df):
    vecs = tfidf_global.transform(df['input'].fillna('').tolist())
    distances, indices = nn_global_3_scores.kneighbors(vecs)
    results = []
    for dist_row, idx_row in zip(distances, indices):
        # pick answer with lowest cosine distance (most similar)
        best_idx = idx_row[np.argmin(dist_row)]
        results.append(train['output'].iloc[best_idx])
    return results

val_pred_e6 = retrieve_top3_most_similar(val)
metrics_e6 = compute_rouge(val_pred_e6, val['output'].tolist())
print(f"\n Experiment 6 — Top-3 Retrieval, Most Similar")
print(f"   ROUGE-1 F1: {metrics_e6['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e6['rougeL_f1']:.4f}")

test_pred_e6 = retrieve_top3_most_similar(test)
make_submission(test['ID'].values, test_pred_e6, '/content/drive/MyDrive/submission_e6_top3_similar.csv')


 Experiment 6 — Top-3 Retrieval, Most Similar
   ROUGE-1 F1: 0.4278
   ROUGE-L F1: 0.3741
Saved: /content/drive/MyDrive/submission_e6_top3_similar.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ..."
1,ID_TS_Aka_Gha_1C80317F,"Yiw, aman pii wɔ mmara anaa nhyehyɛe ahorow a ...","Yiw, aman pii wɔ mmara anaa nhyehyɛe ahorow a ...","Yiw, aman pii wɔ mmara anaa nhyehyɛe ahorow a ..."
2,ID_TS_Aka_Gha_06671AD1,"Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh..."
3,ID_TS_Aka_Gha_BDD640FB,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...
4,ID_TS_Aka_Gha_46685257,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...,Kushiriki ngono na mtu ambaye ana Ukimwi bila ...
2614,ID_TS_Swa_Ken_FD034ADA,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...
2616,ID_TS_Swa_Ken_8DDCE719,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...


## GPU Memory Check

Quick check of available VRAM before loading mT5-base — used to confirm enough memory is
free and to debug OOM errors encountered when fine-tuning the larger model.

In [ ]:
import torch

total = torch.cuda.get_device_properties(0).total_memory / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
allocated = torch.cuda.memory_allocated(0) / 1024**3
free = total - reserved

print(f"Total VRAM:     {total:.2f} GB")
print(f"Allocated:      {allocated:.2f} GB")
print(f"Reserved:       {reserved:.2f} GB")
print(f"Free:           {free:.2f} GB")


Total VRAM:     14.56 GB
Allocated:      0.00 GB
Reserved:       0.00 GB
Free:           14.56 GB


## Experiment 8 — Train+Val Combined Retrieval

Expands the retrieval pool by adding Val to the index (36,501 examples total), giving more
candidates to match test questions against.

**Important fix:** an earlier version of this experiment built the index on train+val and
evaluated on val itself — this caused data leakage (val questions matched themselves exactly,
giving a false 0.9966 ROUGE). This version separates the two: local evaluation uses a
train-only index (honest score), while the test submission uses the train+val index, which is
valid since test questions are genuinely unseen by both.

**Result:** ROUGE-1 0.4278 (local, honest) | Zindi 0.517886 — new best score. Confirms more
data in the retrieval pool directly improves test performance.

In [ ]:
# Experiment 7 — Train+Val Combined Retrieval (CORRECTED)
# We only add val data to the INDEX for test prediction
# but evaluate locally using only train as the index

# For LOCAL evaluation — index on train only (no leakage)
tfidf_eval = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=200_000,
    lowercase=False
)
train_eval_vecs = tfidf_eval.fit_transform(train['input'].fillna('').tolist())
nn_eval = NearestNeighbors(n_neighbors=1, metric='cosine')
nn_eval.fit(train_eval_vecs)

val_pred_e7_eval = []
for text in val['input'].fillna('').tolist():
    vec = tfidf_eval.transform([text])
    _, idx = nn_eval.kneighbors(vec)
    val_pred_e7_eval.append(train['output'].iloc[idx[0][0]])

metrics_e7 = compute_rouge(val_pred_e7_eval, val['output'].tolist())
print(f"\n Experiment 7 — Train+Val Combined (Corrected Eval)")
print(f"   ROUGE-1 F1: {metrics_e7['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e7['rougeL_f1']:.4f}")

# For TEST SUBMISSION — now we can safely add val to the index
# since test questions are genuinely unseen
all_data = pd.concat([train, val], ignore_index=True)

tfidf_all = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=200_000,
    lowercase=False
)
all_vecs = tfidf_all.fit_transform(all_data['input'].fillna('').tolist())
nn_all = NearestNeighbors(n_neighbors=1, metric='cosine')
nn_all.fit(all_vecs)

test_pred_e7 = []
for text in test['input'].fillna('').tolist():
    vec = tfidf_all.transform([text])
    _, idx = nn_all.kneighbors(vec)
    test_pred_e7.append(all_data['output'].iloc[idx[0][0]])

make_submission(test['ID'].values, test_pred_e7, '/content/drive/MyDrive/submission_e7_combined_fixed.csv')


📊 Experiment 7 — Train+Val Combined (Corrected Eval)
   ROUGE-1 F1: 0.4278
   ROUGE-L F1: 0.3741
Saved: /content/drive/MyDrive/submission_e7_combined_fixed.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ..."
1,ID_TS_Aka_Gha_1C80317F,Mmabun betumi anya nneɛma te sɛ: Nkyerɛkyerɛ n...,Mmabun betumi anya nneɛma te sɛ: Nkyerɛkyerɛ n...,Mmabun betumi anya nneɛma te sɛ: Nkyerɛkyerɛ n...
2,ID_TS_Aka_Gha_06671AD1,"Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh..."
3,ID_TS_Aka_Gha_BDD640FB,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...
4,ID_TS_Aka_Gha_46685257,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...,Ɛbue kwan anaa ɛto mu gyina amanyɔsɛm so. Nhye...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,"Maambukizi ya zinaa (STIs), pia yanajulikana k...","Maambukizi ya zinaa (STIs), pia yanajulikana k...","Maambukizi ya zinaa (STIs), pia yanajulikana k..."
2614,ID_TS_Swa_Ken_FD034ADA,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...
2616,ID_TS_Swa_Ken_8DDCE719,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...


## Experiment Summary Table

Consolidated view of all experiments run so far — local ROUGE scores, Zindi public scores,
and key takeaways from each. Used to track progression and decide direction for remaining experiments.

In [ ]:
experiment_log = [
    {"exp_id": "1", "name": "Dummy Baseline", "model": "None", "rouge1": None, "rougeL": None,
     "notes": "Copied input as answer. Validates submission format only. Score not recorded."},
    {"exp_id": "2", "name": "mT5-Small Zero-Shot", "model": "google/mt5-small", "rouge1": None, "rougeL": None,
     "notes": "Raw pretrained model, no fine-tuning. Outputs <extra_id_0> fragments. Score not recorded."},
    {"exp_id": "3", "name": "Global TF-IDF Retrieval", "model": "TF-IDF", "rouge1": 0.4279, "rougeL": 0.3743,
     "zindi_score": 0.497641,
     "notes": "Character n-gram (3,5) TF-IDF, global index across all languages. Best approach so far."},
    {"exp_id": "4", "name": "Language-Specific TF-IDF", "model": "TF-IDF", "rouge1": 0.4209, "rougeL": 0.3656,
     "zindi_score": 0.447403,
     "notes": "Separate TF-IDF index per language subset. Slightly worse — smaller per-language pools hurt retrieval quality."},
    {"exp_id": "5", "name": "Top-3 Longest Answer", "model": "TF-IDF", "rouge1": 0.3713, "rougeL": 0.3085,
     "zindi_score": 0.491549,
     "notes": "Top-3 retrieval, pick longest answer. Lower ROUGE but higher Zindi than expected — LLM judge rewards longer answers."},
    {"exp_id": "6", "name": "Top-3 Most Similar", "model": "TF-IDF", "rouge1": 0.4279, "rougeL": 0.3743,
     "zindi_score": 0.497641,
     "notes": "Top-3 retrieval, pick most similar. Mathematically identical to Top-1 — collapses to Experiment 3."},
    {"exp_id": "7", "name": "Train+Val Combined (Fixed)", "model": "TF-IDF", "rouge1": 0.4278, "rougeL": 0.3741,
     "zindi_score": None,
     "notes": "Index built on train+val for test predictions. Local eval uses train only to prevent leakage. Original version had data leakage (0.9966 local ROUGE) — fixed by separating eval and submission indexes. Submit tomorrow."},
]

exp_df = pd.DataFrame(experiment_log)
display(exp_df[['exp_id', 'name', 'model', 'rouge1', 'rougeL', 'zindi_score', 'notes']])

,exp_id,name,model,rouge1,rougeL,zindi_score,notes
0,1,Dummy Baseline,None,NaN,NaN,NaN,Copied input as answer. Validates submission f...
1,2,mT5-Small Zero-Shot,google/mt5-small,NaN,NaN,NaN,"Raw pretrained model, no fine-tuning. Outputs ..."
2,3,Global TF-IDF Retrieval,TF-IDF,0.4279,0.3743,0.497641,"Character n-gram (3,5) TF-IDF, global index ac..."
3,4,Language-Specific TF-IDF,TF-IDF,0.4209,0.3656,0.447403,Separate TF-IDF index per language subset. Sli...
4,5,Top-3 Longest Answer,TF-IDF,0.3713,0.3085,0.491549,"Top-3 retrieval, pick longest answer. Lower RO..."
5,6,Top-3 Most Similar,TF-IDF,0.4279,0.3743,0.497641,"Top-3 retrieval, pick most similar. Mathematic..."
6,7,Train+Val Combined (Fixed),TF-IDF,0.4278,0.3741,NaN,Index built on train+val for test predictions....


## Install Dependencies for mT5-Base

Additional packages needed for loading and fine-tuning mT5-base in the next experiment.

In [ ]:
!pip install sentencepiece protobuf -q
print("Done")

Done


## Experiment 9 — Load mT5-Base

Loading the larger `google/mt5-base` (580M params, ~2x mT5-small) to test whether model size
closes the gap with TF-IDF retrieval. `expandable_segments` reduces GPU memory fragmentation,
needed after hitting OOM errors with the default allocator on this larger model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float32).to(device)

print(f" {model_name} loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

Device: cuda


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

 google/mt5-base loaded!
Parameters: 582M


## Tokenize with Language-Aware Prompts

Unlike Experiment 3, inputs are wrapped with an explicit language instruction
(`"Answer this health question in Akan: {question}"`), so the model is told which language
to respond in rather than inferring it. Padding tokens in labels are masked with `-100` so
they don't contribute to the loss calculation.

In [ ]:
from datasets import Dataset

max_input_length = 128
max_target_length = 384

SUBSET_TO_LANGUAGE = {
    'Eng': 'English', 'Aka': 'Akan',
    'Lug': 'Luganda', 'Swa': 'Swahili', 'Amh': 'Amharic',
}

def build_prompt(question, subset=None):
    if subset:
        lang = SUBSET_TO_LANGUAGE.get(subset.split('_')[0], 'English')
        return f'Answer this health question in {lang}: {question}'
    return f'Answer this health question: {question}'

def preprocess_function(examples):
    inputs = [build_prompt(q, s) for q, s in zip(examples['input'], examples['subset'])]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(examples['output'], max_length=max_target_length, truncation=True)
    model_inputs['labels'] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels['input_ids']
    ]
    return model_inputs

train_dataset = Dataset.from_pandas(train[['input', 'output', 'subset']])
val_dataset   = Dataset.from_pandas(val[['input', 'output', 'subset']])

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val   = val_dataset.map(preprocess_function, batched=True)

print(tokenized_train)

Map:   0%|          | 0/29815 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'subset', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 29815
})


## Train mT5-Base and Save

Fine-tunes mT5-base for 1 epoch. Batch size reduced to 2 (with gradient accumulation of 4 to
keep the effective batch at 8) and gradient checkpointing enabled to fit the larger model in
T4 VRAM after hitting OOM at the original batch size. `bf16`/`fp16` is auto-detected to avoid
the loss collapsing to NaN, which happened with forced fp16 on mT5-small. Training and saving
to Drive happen in the same cell so a Colab disconnect right after training can't lose the
checkpoint, as happened previously.

**Result:** Training loss 8.886, Validation loss 1.846 (lower than mT5-small's 2.279).

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import shutil

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/mt5_base_finetuned",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=2,       # reduced from 4
    per_device_eval_batch_size=2,        # reduced from 4
    gradient_accumulation_steps=4,       # increased to keep effective batch=8
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True,
    bf16=(device == "cuda" and torch.cuda.is_bf16_supported()),
    fp16=(device == "cuda" and not torch.cuda.is_bf16_supported()),
    logging_steps=100,
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
    generation_max_length=256,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

# Save immediately
shutil.rmtree('/content/drive/MyDrive/mt5_base_finetuned_e8', ignore_errors=True)
model.save_pretrained('/content/drive/MyDrive/mt5_base_finetuned_e8')
tokenizer.save_pretrained('/content/drive/MyDrive/mt5_base_finetuned_e8')
print(" mT5-base saved to Drive!")

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,8.886950,1.846049


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 mT5-base saved to Drive!


## Experiment 9 — Inference & Evaluation

Generates predictions on both Test (for submission) and Val (for local ROUGE) using beam
search instead of greedy decoding — `num_beams=4`, `no_repeat_ngram_size=3`, and
`repetition_penalty=1.3` directly target the repetition loops seen in Experiment 3's output
(e.g. "No. No. No."). Tests whether a bigger model with better prompting and decoding can
close the gap with TF-IDF retrieval's 0.517886 best score.

**Result:**

In [ ]:
import re, csv
from tqdm import tqdm

model.eval()
predictions_e8 = []

for i, row in tqdm(test.iterrows(), total=len(test)):
    prompted = build_prompt(row['input'], row['subset'])
    inputs = tokenizer(prompted, return_tensors="pt", truncation=True, max_length=max_input_length).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions_e8.append(decoded)

# Local ROUGE evaluation
val_preds_e8 = []
for i, row in tqdm(val.iterrows(), total=len(val)):
    prompted = build_prompt(row['input'], row['subset'])
    inputs = tokenizer(prompted, return_tensors="pt", truncation=True, max_length=max_input_length).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    val_preds_e8.append(decoded)

metrics_e8 = compute_rouge(val_preds_e8, val['output'].tolist())
print(f"\n Experiment 8 — mT5-base Fine-Tuned")
print(f"   ROUGE-1 F1: {metrics_e8['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e8['rougeL_f1']:.4f}")

make_submission(test['ID'].values, predictions_e8, '/content/drive/MyDrive/submission_e8_mt5base.csv')

100%|██████████| 6686/6686 [26:33<00:00,  4.20it/s]



 Experiment 8 — mT5-base Fine-Tuned
   ROUGE-1 F1: 0.0132
   ROUGE-L F1: 0.0126
Saved: /content/drive/MyDrive/submission_e8_mt5base.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,a nsɛ,a nsɛ,a nsɛ
1,ID_TS_Aka_Gha_1C80317F,sɛ wonya nipadu,sɛ wonya nipadu,sɛ wonya nipadu
2,ID_TS_Aka_Gha_06671AD1,a nnipa bi mu?,a nnipa bi mu?,a nnipa bi mu?
3,ID_TS_Aka_Gha_BDD640FB,a ɛwɔ ASRH,a ɛwɔ ASRH,a ɛwɔ ASRH
4,ID_TS_Aka_Gha_46685257,nsesaeɛ,nsesaeɛ,nsesaeɛ
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,", mtu mwenye Ukimwi?",", mtu mwenye Ukimwi?",", mtu mwenye Ukimwi?"
2614,ID_TS_Swa_Ken_FD034ADA,"Swahili: Je,...","Swahili: Je,...","Swahili: Je,..."
2615,ID_TS_Swa_Ken_81F38DD4,this health question in Swahili:,this health question in Swahili:,this health question in Swahili:
2616,ID_TS_Swa_Ken_8DDCE719,", tatizo kubwa.",", tatizo kubwa.",", tatizo kubwa."


## Reload Saved mT5-Base

Loads the already fine-tuned mT5-base directly from Drive instead of retraining — used in
later sessions (and on different Colab accounts via the shared Drive folder) to run inference
without repeating the ~3 hour training step.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Force reload from the SAVED checkpoint, not from HuggingFace
model = AutoModelForSeq2SeqLM.from_pretrained('/content/drive/MyDrive/mt5_base_finetuned_e8').to(device)
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/mt5_base_finetuned_e8')
model.eval()

print("Loaded from:", model.name_or_path)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded from: /content/drive/MyDrive/mt5_base_finetuned_e8


## Experiment 9 — Test Set Inference (Reload Path)

Generates test predictions only (no val/local ROUGE pass) using the reloaded model — a
lighter-weight version of the inference cell above, used when running on a fresh session
where only the submission file is needed.

In [ ]:
from tqdm import tqdm
import csv

predictions_e9 = []
model.eval()

for i, row in tqdm(test.iterrows(), total=len(test)):
    prompted = build_prompt(row['input'], row['subset'])
    inputs = tokenizer(prompted, return_tensors="pt", truncation=True, max_length=max_input_length).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions_e9.append(decoded)

make_submission(test['ID'].values, predictions_e9, '/content/drive/MyDrive/submission_e9_mt5base_correct.csv')
print("Experiment 9 done!")

100%|██████████| 2618/2618 [10:27<00:00,  4.17it/s]

Saved: /content/drive/MyDrive/submission_e9_mt5base_correct.csv | Shape: (2618, 4)
Experiment 9 done!


## Experiment 10 — TF-IDF N-gram Range Tuning

All prior TF-IDF experiments used `ngram_range=(3,5)` without testing whether this was optimal.
This sweeps several character n-gram ranges — (1,3), (2,4), (4,6), (1,5) — evaluated locally
against the (3,5) baseline, to find the best-performing configuration before building a final submission.

**Result:** `ngram_range=(2,4)` performed best (ROUGE-1 0.4361, ROUGE-L 0.3826), outperforming
the (3,5) baseline (ROUGE-1 0.4279). Shorter n-grams capture more granular morphological
patterns, which appears to benefit retrieval on morphologically rich languages like Akan and
Luganda. Longer n-grams (4,6) performed worst, likely too specific to find valid partial matches.

In [ ]:
# Experiment 10 — TF-IDF N-gram Range Tuning
ngram_configs = [(1,3), (2,4), (4,6), (1,5)]
results_e10 = {}

for ngram_range in ngram_configs:
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=ngram_range, max_features=200_000, lowercase=False)
    train_vecs_test = vec.fit_transform(train['input'].fillna('').tolist())

    nn = NearestNeighbors(n_neighbors=1, metric='cosine')
    nn.fit(train_vecs_test)

    val_vecs_test = vec.transform(val['input'].fillna('').tolist())
    _, indices = nn.kneighbors(val_vecs_test)
    preds = [train['output'].iloc[idx[0]] for idx in indices]

    metrics = compute_rouge(preds, val['output'].tolist())
    results_e10[ngram_range] = metrics
    print(f"ngram_range={ngram_range}: ROUGE-1={metrics['rouge1_f1']:.4f}, ROUGE-L={metrics['rougeL_f1']:.4f}")

print("\nCurrent baseline (3,5): ROUGE-1=0.4279, ROUGE-L=0.3743")

ngram_range=(1, 3): ROUGE-1=0.4354, ROUGE-L=0.3811
ngram_range=(2, 4): ROUGE-1=0.4357, ROUGE-L=0.3821
ngram_range=(4, 6): ROUGE-1=0.4159, ROUGE-L=0.3621
ngram_range=(1, 5): ROUGE-1=0.4330, ROUGE-L=0.3788

Current baseline (3,5): ROUGE-1=0.4279, ROUGE-L=0.3743


## Experiment 10 — Final Submission: Best N-gram + Train+Val Combined

Combines the two strongest findings from earlier experiments: the tuned `(2,4)` n-gram range
and the enlarged train+val retrieval index (from Experiment 8). Local evaluation uses a
train-only index to avoid the leakage issue identified earlier; the test submission uses the
full train+val index.

**Result:** ROUGE-1 0.4361 (local) | **Zindi score: 0.524792** — best score across all ten experiments.

In [ ]:
# Experiment 10 — Best N-gram (2,4) + Train+Val Combined
all_data = pd.concat([train, val], ignore_index=True)

tfidf_e10 = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=200_000,
    lowercase=False
)
all_vecs_e10 = tfidf_e10.fit_transform(all_data['input'].fillna('').tolist())

nn_e10 = NearestNeighbors(n_neighbors=1, metric='cosine')
nn_e10.fit(all_vecs_e10)

# Local eval (train-only index, no leakage)
tfidf_e10_eval = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=200_000, lowercase=False)
train_vecs_eval = tfidf_e10_eval.fit_transform(train['input'].fillna('').tolist())
nn_e10_eval = NearestNeighbors(n_neighbors=1, metric='cosine')
nn_e10_eval.fit(train_vecs_eval)

val_vecs_eval = tfidf_e10_eval.transform(val['input'].fillna('').tolist())
_, indices_eval = nn_e10_eval.kneighbors(val_vecs_eval)
val_preds_e10 = [train['output'].iloc[idx[0]] for idx in indices_eval]

metrics_e10 = compute_rouge(val_preds_e10, val['output'].tolist())
print(f"Experiment 10 — N-gram (2,4) + Train+Val Combined")
print(f"   ROUGE-1 F1: {metrics_e10['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1: {metrics_e10['rougeL_f1']:.4f}")

# Test submission (train+val index)
test_vecs_e10 = tfidf_e10.transform(test['input'].fillna('').tolist())
_, indices_test = nn_e10.kneighbors(test_vecs_e10)
test_preds_e10 = [all_data['output'].iloc[idx[0]] for idx in indices_test]

make_submission(test['ID'].values, test_preds_e10, '/content/drive/MyDrive/submission_e10_ngram24_combined.csv')

Experiment 10 — N-gram (2,4) + Train+Val Combined
   ROUGE-1 F1: 0.4357
   ROUGE-L F1: 0.3821
Saved: /content/drive/MyDrive/submission_e10_ngram24_combined.csv | Shape: (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa ..."
1,ID_TS_Aka_Gha_1C80317F,Mmabun betumi anya nneɛma te sɛ: Nkyerɛkyerɛ n...,Mmabun betumi anya nneɛma te sɛ: Nkyerɛkyerɛ n...,Mmabun betumi anya nneɛma te sɛ: Nkyerɛkyerɛ n...
2,ID_TS_Aka_Gha_06671AD1,"Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh...","Dadwen, adwenemhaw, ne obu a wonni yɛ kɛse. Ɔh..."
3,ID_TS_Aka_Gha_BDD640FB,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...,Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwum...
4,ID_TS_Aka_Gha_46685257,Ɛma wosusuw nsɛm ho ansa na woayi bi. Nsɛm a ɛ...,Ɛma wosusuw nsɛm ho ansa na woayi bi. Nsɛm a ɛ...,Ɛma wosusuw nsɛm ho ansa na woayi bi. Nsɛm a ɛ...
...,...,...,...,...
2613,ID_TS_Swa_Ken_C9525185,"Maambukizi ya zinaa (STIs), pia yanajulikana k...","Maambukizi ya zinaa (STIs), pia yanajulikana k...","Maambukizi ya zinaa (STIs), pia yanajulikana k..."
2614,ID_TS_Swa_Ken_FD034ADA,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...,HAPANA. Mimea na dawa mbadala haziwezi kuponya...
2615,ID_TS_Swa_Ken_81F38DD4,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...,Ikiwa unashuku kuwa mpenzi wako anaweza kuwa n...
2616,ID_TS_Swa_Ken_8DDCE719,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...,Kunusurika na kudhibiti Ukimwi/UKIMWI kunaweze...


## Fixing Missing Values in Submission File

This cell loads the submission CSV from Google Drive and fills in missing values in the target columns (`TargetRLF1`, `TargetR1F1`, and `TargetLLM`) with a default placeholder, "No answer available." This ensures there are no NaN values that could break evaluation or submission formatting. The cleaned file is then saved as a new CSV, keeping the original intact while producing a version that is safe to use.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/submission_e8_mt5base.csv')
df['TargetRLF1'] = df['TargetRLF1'].fillna("No answer available.")
df['TargetR1F1'] = df['TargetR1F1'].fillna("No answer available.")
df['TargetLLM']  = df['TargetLLM'].fillna("No answer available.")

df.to_csv('/content/drive/MyDrive/submission_e9_mt5base_fixed.csv', index=False)
print("Fixed and saved!")

Fixed and saved!


## Quick Inference Check

Takes one example, formats it into the expected prompt, tokenizes it, and runs it through the model to generate a response using beam search with repetition controls. The output is then decoded into text and printed to quickly verify the model is producing sensible results.

In [ ]:
test_q = train['input'].iloc[0]
test_subset = train['subset'].iloc[0]
prompted = build_prompt(test_q, test_subset)

inputs = tokenizer(prompted, return_tensors="pt", truncation=True, max_length=max_input_length).to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=max_target_length,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        early_stopping=True
    )
print("Output:", tokenizer.decode(outputs[0], skip_special_tokens=True))

## Inspecting a Specific Example

Retrieves the row with the given ID from the test set, prints its input text, and shows its length to check how long the sequence is.

In [ ]:
problem_row = test[test['ID'] == 'ID_TS_Eng_Gha_89D7FD6C']
print(problem_row['input'].values[0])
print("\nLength:", len(problem_row['input'].values[0]))

Delete

Length: 6


In [ ]:
import pandas as pd

sub = pd.read_csv('/content/drive/MyDrive/submission_e9_mt5base_correct.csv')

print("Shape:", sub.shape)
print("Columns:", sub.columns.tolist())
print("\nNull values:")
print(sub.isna().sum())
print("\nEmpty strings:")
print((sub['TargetRLF1'].str.strip() == '').sum())
print("\nDuplicate IDs:", sub['ID'].duplicated().sum())

In [ ]:
from tqdm import tqdm
import json

final_predictions = []
model.eval()

checkpoint_path = '/content/drive/MyDrive/e9_predictions_checkpoint.json'

for i, row in tqdm(test.iterrows(), total=len(test)):
    prompted = build_prompt(row['input'], row['subset'])
    inputs = tokenizer(prompted, return_tensors="pt", truncation=True, max_length=max_input_length).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    final_predictions.append(decoded)

    # Save a checkpoint to Drive every 200 rows so a disconnect can't lose everything
    if (len(final_predictions)) % 200 == 0:
        with open(checkpoint_path, 'w') as f:
            json.dump(final_predictions, f)

print("DONE — total predictions:", len(final_predictions))
print("First 3:", final_predictions[:3])

# Final save — both checkpoint and submission file
with open(checkpoint_path, 'w') as f:
    json.dump(final_predictions, f)

final_clean = [p if p.strip() else "No answer available." for p in final_predictions]
make_submission(test['ID'].values, final_clean, '/content/drive/MyDrive/submission_e9_REAL.csv')
print("SUBMISSION SAVED TO DRIVE!")

100%|██████████| 2618/2618 [1:57:59<00:00,  2.70s/it]

DONE — total predictions: 2618
First 3: ['Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow ho nsɛm a ɛwɔ GBV ano ma.', 'Nneɛma a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu no mu aba: Mmabun a wɔde wɔn ho bɛhyɛ nsɛm a ɛfa nna ho nkyerɛkyerɛ ne nneyɛe a wobenya no so mu aba, na wɔayɛ hokwan ahorow ho nkuran.', "Mmabun bɛtumi afa ehunu nsusuanso a ɛtumi aba nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a asɛm bi mu ayɛ den, nneɛma bɔne a wosiw ano, ne nna mu akwahosan ho adwumayɛbea ahorow so. Nneɛma a Wɔbɛhwehwɛ Mmoa a Wobɛfa So: Sɛ wobɛfa so nkɔmmɔbɔ anaa nsɛm a wode di dwuma de nkitahodi a ebetumi aboa ma wɔde bɛma wɔn ho bɛhyɛ mu."]
Saved: /content/drive/MyDrive/submission_e9_REAL.csv | Shape: (2618, 4)
SUBMISSION SAVED TO DRIVE!


In [ ]:
print("Model loaded from:", model.name_or_path)

Model loaded from: /content/drive/MyDrive/mt5_base_finetuned_e8


In [9]:
import json

notebook_path = '/content/drive/MyDrive/Colab Notebooks/Emmanuella Briggs_Summative MLT1.ipynb'

with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Remove the broken widgets metadata
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Also fix any cell-level widget state issues
for cell in nb.get('cells', []):
    if 'widgets' in cell.get('metadata', {}):
        del cell['metadata']['widgets']

with open(notebook_path, 'w') as f:
    json.dump(nb, f, indent=1)

print("Fixed! Re-download and push to GitHub.")

Fixed! Re-download and push to GitHub.
